In [ ]:
!pip install evaluate
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [ ]:
!pip install datasets

from datasets import load_dataset
import pandas as pd

# Load full dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

# 🔥 CHANGE SAMPLE SIZE HERE
train_size = 5000
val_size = 1500

train_df = pd.DataFrame(dataset["train"][:train_size])
val_df = pd.DataFrame(dataset["validation"][:val_size])

# Keep only needed columns
train_df = train_df[["article", "highlights"]]
val_df = val_df[["article", "highlights"]]

# Save NEW CSVs (overwrite old ones)
train_df.to_csv("train_subset.csv", index=False)
val_df.to_csv("val_subset.csv", index=False)

print("Updated dataset created!")
print("Train size:", len(train_df))
print("Validation size:", len(val_df))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Updated dataset created!
Train size: 5000
Validation size: 1500


In [ ]:
train_df = pd.read_csv("/content/train_subset.csv")
val_df = pd.read_csv("/content/val_subset.csv")

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],  # attention projections
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7876


In [ ]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["highlights"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./t5_lora_news",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    eval_strategy="no",
    save_strategy="epoch",
    num_train_epochs=4,
    learning_rate=1e-4,
    fp16=True,
    #load_best_model_at_end=True,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
!pip install rouge_score
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=True
    )

    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {k: round(v, 4) for k, v in result.items()}

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=31d9aabe884c04dd5c283808c1d20aa5c8953e768e2c01b6b6466efc56195fb7
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

model.save_pretrained("./t5_lora_news_adapter")
tokenizer.save_pretrained("./t5_lora_news_adapter")

print("Training Complete!")

Step,Training Loss
500,13.091654
1000,6.736106
1500,6.600734
2000,6.648806
2500,6.572887


Training Complete!


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq
import evaluate

rouge = evaluate.load("rouge")

model.eval()
model.to("cuda")

eval_dataset = val_dataset.select(range(200))
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

dataloader = DataLoader(
    eval_dataset,
    batch_size=2,
    collate_fn=data_collator
)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in dataloader:
        batch = {k: v.to("cuda") for k, v in batch.items()}

        generated_ids = model.generate(
          input_ids=batch["input_ids"],
          attention_mask=batch["attention_mask"],
          max_length=100,
          min_length=30,
          num_beams=4,
          length_penalty=1.5,
          no_repeat_ngram_size=3,
          early_stopping=True
        )

        preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        labels = batch["labels"]
        labels = torch.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        all_preds.extend(preds)
        all_labels.extend(decoded_labels)

        torch.cuda.empty_cache()

results = rouge.compute(predictions=all_preds, references=all_labels)
print(results)

{'rouge1': np.float64(0.3585023278844744), 'rouge2': np.float64(0.16370411403962803), 'rougeL': np.float64(0.268117947327462), 'rougeLsum': np.float64(0.26846020342297716)}


In [ ]:
print("\n--- SAMPLE GENERATED SUMMARIES ---\n")

num_samples = 5   # you can change to 10

for i in range(num_samples):

    sample = val_df.iloc[i]

    article = sample["article"]
    reference_summary = sample["highlights"]

    # Prepare input
    input_text = "summarize: " + article
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to("cuda")

    # Generate summary
    summary_ids = model.generate(
        input_ids=inputs["input_ids"],
        max_length=356,
        min_length=100,
        num_beams=8,
        length_penalty=1.8,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    print("ARTICLE:\n", article[:500])
    print("\nREFERENCE SUMMARY:\n", reference_summary)
    print("\nGENERATED SUMMARY:\n", generated_summary)
    print("\n---------------------------------\n")


--- SAMPLE GENERATED SUMMARIES ---

ARTICLE:
 (CNN)Share, and your gift will be multiplied. That may sound like an esoteric adage, but when Zully Broussard selflessly decided to give one of her kidneys to a stranger, her generosity paired up with big data. It resulted in six patients receiving transplants. That surprised and wowed her. "I thought I was going to help this one person who I don't know, but the fact that so many people can have a life extension, that's pretty big," Broussard told CNN affiliate KGO. She may feel guided in her ge

REFERENCE SUMMARY:
 Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

GENERATED SUMMARY:
 Zully Broussard gave one of her kidneys to a stranger . Her generosity paired up with big data, and six patients received transplants . "I know this entire journey is much bigger than all of us," she wrote on Facebook . The power that multiplied her gift was data pro

In [ ]:
from transformers import AutoModelForSeq2SeqLM

baseline_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to("cuda")
baseline_model.eval()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
import pandas as pd
import os
from datasets import load_dataset # Import load_dataset for re-creation
from transformers import T5Tokenizer # Import T5Tokenizer

# Define the val_size if the file needs to be re-created
val_size = 800 # This should match the size used in the dataset creation cell (fpvwAPmKl43i)

# Check if val_subset.csv exists, if not, create it
if not os.path.exists("/content/val_subset.csv"):
    print("Warning: val_subset.csv not found. Re-creating it...")
    try:
        # Load full dataset to re-create the validation subset
        dataset_full = load_dataset("cnn_dailymail", "3.0.0")
        val_df = pd.DataFrame(dataset_full["validation"][:val_size]) # Assign directly to val_df
        val_df = val_df[["article", "highlights"]]
        val_df.to_csv("/content/val_subset.csv", index=False)
        print("val_subset.csv re-created successfully.")
    except Exception as e:
        print(f"Error re-creating val_subset.csv: {e}")
        raise # Re-raise the error if creation fails
else:
    val_df = pd.read_csv("/content/val_subset.csv") # Only read if it exists

# Initialize tokenizer for this cell's scope
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)

baseline_predictions = []

for sample in val_df["article"][:500]:

    input_text = "summarize: " + sample

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to("cuda")

    summary_ids = baseline_model.generate(
        inputs["input_ids"],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    pred = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    baseline_predictions.append(pred)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
baseline_references = val_df["highlights"][:500].tolist()

In [ ]:
!pip install evaluate
!pip install rouge_score
import evaluate

rouge = evaluate.load("rouge")

baseline_results = rouge.compute(
    predictions=baseline_predictions,
    references=baseline_references
)

print("Baseline Results:")
print(baseline_results)

Baseline Results:
{'rouge1': np.float64(0.28365587826110367), 'rouge2': np.float64(0.11083855968526969), 'rougeL': np.float64(0.21377288351054202), 'rougeLsum': np.float64(0.24536934172722114)}


In [ ]:
!rm -rf t5_lora_news
!rm -rf t5_lora_news_adapter

import torch
torch.cuda.empty_cache()

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
!rm -rf ./t5_lora_news

In [ ]:
!rm -rf ~/.cache/huggingface


In [ ]:
!pip install streamlit
!pip install transformers
!pip install peft
!pip install sentencepiece
!pip install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 142.9 MB/s eta 0:00:00


In [ ]:
!ls

sample_data   t5_lora_news_adapter  val_subset.csv
t5_lora_news  train_subset.csv


In [ ]:
!ls t5_lora_news_adapter


adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  tokenizer_config.json


In [ ]:
import streamlit as st
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, AutoModelForSeq2SeqLM
from peft import PeftModel
from newspaper import Article

st.title("LORYX: LoRA vs Baseline Summarizer")

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------- LOAD MODELS --------
@st.cache_resource
def load_model():
    base_model = T5ForConditionalGeneration.from_pretrained("t5-base")
    tokenizer = T5Tokenizer.from_pretrained("t5-base")

    lora_model = PeftModel.from_pretrained(base_model, "t5_lora_news_adapter")
    lora_model.to(device).eval()

    baseline_model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")
    baseline_model.to(device).eval()

    return lora_model, baseline_model, tokenizer

lora_model, baseline_model, tokenizer = load_model()

# -------- EXTRACT TEXT FROM URL --------
def extract_text(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except:
        return None

# -------- SUMMARIZATION FUNCTION --------
def generate_summary(model, text):
    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    output = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        min_length=40,
        num_beams=6,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# -------- INPUT SECTION --------
option = st.radio("Choose Input Type", ["Text", "URL"])

if option == "Text":
    user_input = st.text_area("Enter Article Text")

else:
    user_input = st.text_input("Paste Article URL")

# -------- BUTTON --------
if st.button("Generate Summary"):

    if not user_input:
        st.warning("Please provide input")
    else:
        # Handle URL
        if option == "URL":
            text = extract_text(user_input)

            if not text:
                st.error("❌ Failed to extract article")
                st.stop()
        else:
            text = user_input

        text = text[:2000]  # important truncation

        # Generate both summaries
        lora_summary = generate_summary(lora_model, text)
        baseline_summary = generate_summary(baseline_model, text)

        # Output
        st.subheader("Fine-Tuned (LoRA) Summary")
        st.write(lora_summary)

        st.subheader("Baseline Summary")
        st.write(baseline_summary)

Writing app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok, conf
import os

# The error 'HTTP Error 403: Forbidden' indicates that pyngrok failed to download
# the ngrok executable automatically from the specified URL. This often happens
# if the download link changes or becomes restricted by ngrok.

# To fix this, you need to:
# 1. Manually download the ngrok executable for your operating system (Linux AMD64 for Colab)
#    from the official ngrok website: https://ngrok.com/download
# 2. Upload the downloaded 'ngrok' executable to your Colab environment (e.g., to /content/ngrok).
#    You can use the file browser on the left in Colab to upload the file.
# 3. Make the downloaded file executable: `!chmod +x /content/ngrok` (if placed in /content/)
# 4. Specify the path to this executable in the 'NGROK_EXECUTABLE_PATH' variable below.

# --- USER ACTION REQUIRED ---
# Replace "/content/ngrok" with the actual path where you uploaded the ngrok executable.
# If you place it directly in /content/, the path would be "/content/ngrok".
NGROK_EXECUTABLE_PATH = "/content/ngrok" # <--- IMPORTANT: Update this path!

# Your ngrok authentication token
NGROK_AUTH_TOKEN = "3B1afcGnF0vcE4asmObyjpLi6dW_3a3CJyEmDkJa3CQeasbFQ"

# Declare pyngrok_config as a global variable
global pyngrok_config

if not os.path.exists(NGROK_EXECUTABLE_PATH):
    print(f"Error: ngrok executable not found at '{NGROK_EXECUTABLE_PATH}'.")
    print("Please manually download, upload, and make executable the ngrok binary, then update NGROK_EXECUTABLE_PATH.")
    print("E.g., after uploading to /content/, run: `!chmod +x /content/ngrok`")
else:
    # Create a PyngrokConfig object with the specified ngrok_path
    # This prevents pyngrok from attempting to download the executable implicitly.
    pyngrok_config = conf.PyngrokConfig(
        auth_token=NGROK_AUTH_TOKEN,
        ngrok_path=NGROK_EXECUTABLE_PATH
    )

    # Set the auth token using the custom configuration
    ngrok.set_auth_token(NGROK_AUTH_TOKEN, pyngrok_config=pyngrok_config)
    print("ngrok auth token set successfully using the specified ngrok executable.")

ngrok auth token set successfully using the specified ngrok executable.


In [ ]:
from pyngrok import ngrok, conf

# Replace 'YOUR_AUTH_TOKEN' with your actual ngrok authtoken
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok.set_auth_token("YOUR_AUTH_TOKEN")

# Ensure pyngrok_config is defined from HfiXcvR0VF7n or re-initialize if needed
# This assumes HfiXcvR0VF7n has been run and pyngrok_config is global.
# If HfiXcvR0VF7n has not been run, you might need to re-initialize it here.

# Re-create pyngrok_config if it's not global (for robustness, though we made it global above)
if 'pyngrok_config' not in globals():
    # This block should ideally not be reached if HfiXcvR0VF7n is run first
    NGROK_AUTH_TOKEN = "3B1afcGnF0vcE4asmObyjpLi6dW_3a3CJyEmDkJa3CQeasbFQ" # Use your actual token
    NGROK_EXECUTABLE_PATH = "/content/ngrok"
    pyngrok_config = conf.PyngrokConfig(
        auth_token=NGROK_AUTH_TOKEN,
        ngrok_path=NGROK_EXECUTABLE_PATH
    )

public_url = ngrok.connect(8501, pyngrok_config=pyngrok_config)
print(public_url)

NgrokTunnel: "https://rhett-nonesthetic-shanae.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!ls -F /content/

app.py	  ngrok*	t5_lora_news/	       train_subset.csv
logs.txt  sample_data/	t5_lora_news_adapter/  val_subset.csv


In [ ]:
!chmod +x /content/ngrok

chmod: cannot access '/content/ngrok': No such file or directory


In [ ]:
!chmod +x /content/ngrok

In [ ]:
!zip -r t5_lora_news_adapter.zip t5_lora_news_adapter

  adding: t5_lora_news_adapter/ (stored 0%)
  adding: t5_lora_news_adapter/tokenizer.json (deflated 79%)
  adding: t5_lora_news_adapter/adapter_model.safetensors (deflated 7%)
  adding: t5_lora_news_adapter/tokenizer_config.json (deflated 83%)
  adding: t5_lora_news_adapter/README.md (deflated 66%)
  adding: t5_lora_news_adapter/adapter_config.json (deflated 58%)
